# Notebook 1 — Variant → Pathway Mapping

**Goal**: Demonstrate end-to-end variant → pathway mapping for monogenic
neurology variants.

## Workflow

```
ClinVar TSV ──┐
              ├──► normalize_variants ──► build_variant_pathway_graph ──► rank_pathways
gnomAD TSV  ──┘
              ▲
KEGG / Reactome ──► build_gene_pathway_map
```

This notebook uses a small **synthetic dataset** so it runs offline without
downloading multi-GB public files.  Replace the synthetic data blocks with
real loaders once the raw files are available under `data/raw/`.

## 1. Setup

In [ ]:
import sys, os
# Ensure repo root is on the path when running the notebook directly
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from data_ingestion.load_clinvar import ClinVarRecord
from data_ingestion.load_pathways import Pathway
from preprocessing.normalize_variants import normalize_clinvar_records
from preprocessing.gene_to_pathway_mapping import build_gene_pathway_map
from models.variant_pathway_model import build_variant_pathway_graph
from models.scoring import rank_pathways

print('Imports OK')

## 2. Synthetic ClinVar data

In production, replace with:
```python
from data_ingestion.load_clinvar import load_clinvar_tsv
clinvar_records = load_clinvar_tsv('data/raw/variant_summary.txt.gz')
```

In [ ]:
clinvar_records = [
    ClinVarRecord(
        variant_id='1388948', gene_symbol='LRRK2', chrom='12',
        pos=40340400, ref='G', alt='A',
        clinical_significance='Pathogenic',
        condition="Parkinson disease, late-onset",
        review_status='criteria provided, single submitter',
    ),
    ClinVarRecord(
        variant_id='4149', gene_symbol='PSEN1', chrom='14',
        pos=73659468, ref='C', alt='T',
        clinical_significance='Pathogenic',
        condition="Alzheimer disease, early-onset",
        review_status='reviewed by expert panel',
    ),
    ClinVarRecord(
        variant_id='54321', gene_symbol='HTT', chrom='4',
        pos=3076407, ref='CAG', alt='CAGCAGCAG',
        clinical_significance='Pathogenic',
        condition="Huntington disease",
        review_status='criteria provided, multiple submitters, no conflicts',
    ),
    ClinVarRecord(
        variant_id='99999', gene_symbol='LRRK2', chrom='12',
        pos=40734202, ref='G', alt='C',
        clinical_significance='Likely pathogenic',
        condition="Parkinson disease, late-onset",
        review_status='criteria provided, single submitter',
    ),
]

print(f'Loaded {len(clinvar_records)} ClinVar records')

## 3. Normalise variants

In [ ]:
variants = normalize_clinvar_records(clinvar_records)

print(f'Normalised {len(variants)} variants')
for v in variants:
    print(f'  {v.key}  gene={v.gene_symbol}  sig={v.clinical_significance}')

## 4. Synthetic pathway data

In production, replace with:
```python
from data_ingestion.load_pathways import load_kegg_pathways
pathways = load_kegg_pathways('hsa', fetch_genes=True)
```

In [ ]:
pathways = [
    Pathway(
        pathway_id='hsa05012',
        name='Parkinson disease - Homo sapiens (human)',
        source='KEGG',
        gene_symbols=['LRRK2', 'PINK1', 'PARK7', 'SNCA', 'UCHL1'],
    ),
    Pathway(
        pathway_id='hsa05010',
        name="Alzheimer disease - Homo sapiens (human)",
        source='KEGG',
        gene_symbols=['PSEN1', 'PSEN2', 'APP', 'APOE', 'BACE1'],
    ),
    Pathway(
        pathway_id='hsa05016',
        name="Huntington disease - Homo sapiens (human)",
        source='KEGG',
        gene_symbols=['HTT', 'BDNF', 'CASP3', 'HDAC4', 'TBP'],
    ),
    Pathway(
        pathway_id='hsa04010',
        name='MAPK signaling pathway - Homo sapiens (human)',
        source='KEGG',
        gene_symbols=['LRRK2', 'MAPK1', 'MAPK3', 'RAF1', 'MAP2K1'],
    ),
]

print(f'Loaded {len(pathways)} pathways')

## 5. Build gene → pathway index

In [ ]:
gene_map = build_gene_pathway_map(pathways)

print(f'Gene→pathway index: {gene_map.gene_count} genes × {gene_map.pathway_count} pathways')

# Inspect LRRK2 pathway membership
from preprocessing.gene_to_pathway_mapping import lookup_pathways_for_gene
lrrk2_pathways = lookup_pathways_for_gene(gene_map, 'LRRK2')
print(f'LRRK2 participates in: {lrrk2_pathways}')

## 6. Build variant → pathway graph

In [ ]:
graph = build_variant_pathway_graph(variants, gene_map)

print(f'Graph: {graph.variant_count} variants linked to {graph.pathway_count} pathways')
print(f'Total edges: {len(graph.links)}')

## 7. Inspect edges

In [ ]:
print('Variant → Pathway edges:')
for link in graph.links:
    print(f'  {link.variant_key}  →  [{link.pathway_id}] {link.pathway_name[:50]}')

## 8. Score and rank pathways

In [ ]:
ranked = rank_pathways(graph)

print('\nRanked pathways by normalised impact score:')
print(f'{"Pathway ID":<15} {"Score":>8} {"Norm score":>12} {"Variants":>10} {"Genes":>7}  Name')
print('-' * 90)
for ps in ranked:
    print(
        f'{ps.pathway_id:<15} {ps.score:>8.3f} {ps.normalised_score:>12.3f}'
        f' {ps.variant_count:>10} {ps.gene_count:>7}  {ps.pathway_name[:45]}'
    )

## 9. Next steps

- Replace synthetic data with real ClinVar / gnomAD downloads
- Expand pathway coverage: add Reactome via `load_reactome_pathways`
- Add HPO-based phenotype filtering for disease-specific analyses
- See `evaluation.ipynb` for quantitative performance metrics